In [1]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.layers import (
    Input, Embedding, Flatten, Dense, Concatenate, 
    Dropout, Dot, LayerNormalization, Add)
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.regularizers import l2
from tensorflow.keras.utils import Sequence
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MultiLabelBinarizer, MinMaxScaler, LabelEncoder
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.feature_extraction.text import TfidfVectorizer
import matplotlib.pyplot as plt
import re
import os
import gc
import joblib
import scipy.sparse as sp

print(f"TensorFlow Version: {tf.__version__}")

BASE_PATH = '../input/movielens-25m-dataset/ml-25m/'
OUTPUT_DIR = '/kaggle/working/'

if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR)
print(f"Output Dir: {OUTPUT_DIR}")

2025-11-18 15:24:34.409475: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1763479474.625550      19 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1763479474.687815      19 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

TensorFlow Version: 2.18.0
Output Dir: /kaggle/working/


In [2]:
try:
    print("Processing Ratings & Movies...")
    ratings_df = pd.read_csv(BASE_PATH + 'ratings.csv')
    movies_df = pd.read_csv(BASE_PATH + 'movies.csv')
    
    movies_df['year'] = movies_df['title'].str.extract(r'\((\d{4})\)')
    movies_df['year'] = pd.to_numeric(movies_df['year'], errors='coerce')
    mean_year = int(movies_df['year'].mean())
    movies_df['year'] = movies_df['year'].fillna(mean_year).astype(int)
    
    scaler = MinMaxScaler()
    movies_df['year_scaled'] = scaler.fit_transform(movies_df[['year']])
    
    movies_df['genres_list'] = movies_df['genres'].apply(lambda x: x.split('|'))
    mlb = MultiLabelBinarizer()
    genres_encoded = mlb.fit_transform(movies_df['genres_list'])
    genre_columns = list(mlb.classes_)
    genres_df = pd.DataFrame(genres_encoded, columns=genre_columns, index=movies_df.index)
    movies_df = pd.concat([movies_df, genres_df], axis=1)

    user_encoder = LabelEncoder()
    movie_encoder = LabelEncoder()
    ratings_df['user_idx'] = user_encoder.fit_transform(ratings_df['userId'])
    ratings_df['movie_idx'] = movie_encoder.fit_transform(ratings_df['movieId'])

    n_users = ratings_df['user_idx'].nunique()
    n_movies = ratings_df['movie_idx'].nunique()
    n_genres = len(genre_columns)
    print(f"Users: {n_users}, Movies: {n_movies}, Genres: {n_genres}")

    movie_id_map = dict(zip(movie_encoder.classes_, movie_encoder.transform(movie_encoder.classes_)))
    movies_df['movie_idx'] = movies_df['movieId'].map(movie_id_map)
    movies_df = movies_df.dropna(subset=['movie_idx'])
    movies_df['movie_idx'] = movies_df['movie_idx'].astype(int)
    
    metadata_cols = ['movie_idx', 'year_scaled'] + genre_columns
    data = pd.merge(
        ratings_df[['user_idx', 'movie_idx', 'rating']], 
        movies_df[metadata_cols], 
        on='movie_idx', how='inner'
    )
    
    train_val_df, test_df = train_test_split(data, test_size=0.1, random_state=42)
    train_df, val_df = train_test_split(train_val_df, test_size=0.1111, random_state=42)
    print(f"Train: {len(train_df)}, Val: {len(val_df)}, Test: {len(test_df)}")

    train_df.to_parquet(OUTPUT_DIR + 'train.parquet')
    val_df.to_parquet(OUTPUT_DIR + 'val.parquet')
    test_df.to_parquet(OUTPUT_DIR + 'test.parquet')
    joblib.dump(user_encoder, OUTPUT_DIR + 'user_encoder.joblib')
    joblib.dump(movie_encoder, OUTPUT_DIR + 'movie_encoder.joblib')
    joblib.dump(scaler, OUTPUT_DIR + 'year_scaler.joblib')
    joblib.dump(mlb, OUTPUT_DIR + 'mlb_encoder.joblib')
    joblib.dump(genre_columns, OUTPUT_DIR + 'genre_columns.joblib')
    
    metadata = {'n_users': n_users, 'n_movies': n_movies, 'n_genres': n_genres}
    joblib.dump(metadata, OUTPUT_DIR + 'metadata.joblib')
    print("Cell 2 Completed")

except Exception as e:
    print(f"Error Cell 2: {e}")
finally:
    try:
        del data, train_df, val_df, test_df, ratings_df, movies_df, train_val_df, genres_df
    except NameError:
        pass
    gc.collect()

Processing Ratings & Movies...
Users: 162541, Movies: 59047, Genres: 20
Train: 20000325, Val: 2499760, Test: 2500010
Cell 2 Completed


In [3]:
try:
    print("Processing Genome...")
    movie_encoder = joblib.load(OUTPUT_DIR + 'movie_encoder.joblib')
    metadata = joblib.load(OUTPUT_DIR + 'metadata.joblib')
    n_movies = metadata['n_movies']

    genome_scores = pd.read_csv(BASE_PATH + 'genome-scores.csv')
    n_tags = genome_scores['tagId'].nunique()
    print(f"Genome Tags: {n_tags}")
    
    movie_id_map = dict(zip(movie_encoder.classes_, movie_encoder.transform(movie_encoder.classes_)))
    genome_scores['movie_idx'] = genome_scores['movieId'].map(movie_id_map)
    genome_scores = genome_scores.dropna(subset=['movie_idx'])
    genome_scores['movie_idx'] = genome_scores['movie_idx'].astype(int)

    data = genome_scores['relevance'].values
    rows = genome_scores['movie_idx'].values
    cols = genome_scores['tagId'].values - 1 
    genome_sparse_matrix = sp.csr_matrix((data, (rows, cols)), shape=(n_movies, n_tags))
    
    sp.save_npz(OUTPUT_DIR + 'genome_sparse_matrix.npz', genome_sparse_matrix)
    
    metadata['n_tags'] = n_tags
    joblib.dump(metadata, OUTPUT_DIR + 'metadata.joblib')
    print("Cell 3 Completed")

except Exception as e:
    print(f"Error Cell 3: {e}")
finally:
    try:
        del genome_scores, genome_sparse_matrix, movie_encoder, metadata
    except NameError:
        pass
    gc.collect()


Processing Genome...
Genome Tags: 1128
Cell 3 Completed


In [4]:
try:
    print("Training Baseline Model...")
    train_df = pd.read_parquet(OUTPUT_DIR + 'train.parquet')
    val_df = pd.read_parquet(OUTPUT_DIR + 'val.parquet')
    test_df = pd.read_parquet(OUTPUT_DIR + 'test.parquet')
    metadata = joblib.load(OUTPUT_DIR + 'metadata.joblib')
    n_users = metadata['n_users']
    n_movies = metadata['n_movies']
    n_genres = metadata['n_genres']
    genre_columns = joblib.load(OUTPUT_DIR + 'genre_columns.joblib')

    def build_baseline_model(n_users, n_movies, n_genres):
        user_input = Input(shape=(1,), name='user_input')
        movie_input = Input(shape=(1,), name='movie_input')
        genre_input = Input(shape=(n_genres,), name='genre_input')
        year_input = Input(shape=(1,), name='year_input')
        
        user_embedding = Embedding(n_users, 64, name='user_embedding')(user_input)
        user_vec = Flatten(name='flatten_user')(user_embedding)
        
        movie_embedding = Embedding(n_movies, 64, name='movie_embedding')(movie_input)
        movie_vec = Flatten(name='flatten_movie')(movie_embedding)
        
        genre_vec = Dense(16, activation='relu', name='genre_dense')(genre_input)
        
        concat = Concatenate()([user_vec, movie_vec, genre_vec, year_input])
        dense_1 = Dense(128, activation='relu')(concat)
        dropout_1 = Dropout(0.3)(dense_1)
        dense_2 = Dense(64, activation='relu')(dropout_1)
        dropout_2 = Dropout(0.3)(dense_2)
        output = Dense(1, activation='linear')(dropout_2)
        
        model = Model(inputs=[user_input, movie_input, genre_input, year_input], outputs=output)
        model.compile(optimizer=Adam(0.001), loss='mse', metrics=[tf.keras.metrics.RootMeanSquaredError(name='rmse'), tf.keras.metrics.MeanAbsoluteError(name='mae')])
        return model

    X_train = {'user_input': train_df['user_idx'], 'movie_input': train_df['movie_idx'], 'genre_input': train_df[genre_columns], 'year_input': train_df['year_scaled']}
    y_train = train_df['rating']
    X_val = {'user_input': val_df['user_idx'], 'movie_input': val_df['movie_idx'], 'genre_input': val_df[genre_columns], 'year_input': val_df['year_scaled']}
    y_val = val_df['rating']

    model_baseline = build_baseline_model(n_users, n_movies, n_genres)
    history_baseline = model_baseline.fit(X_train, y_train, batch_size=8192, epochs=10, verbose=2, validation_data=(X_val, y_val))

    model_baseline.save(OUTPUT_DIR + 'baseline_model.keras')
    pd.DataFrame(history_baseline.history).to_csv(OUTPUT_DIR + 'baseline_history.csv')

    X_test = {'user_input': test_df['user_idx'], 'movie_input': test_df['movie_idx'], 'genre_input': test_df[genre_columns], 'year_input': test_df['year_scaled']}
    y_test = test_df['rating']
    results_baseline = model_baseline.evaluate(X_test, y_test, batch_size=8192)
    print(f"Baseline RMSE: {results_baseline[1]:.4f}")

except Exception as e:
    print(f"Error Cell 4: {e}")
finally:
    try:
        del train_df, val_df, test_df, X_train, y_train, X_val, y_val, X_test, y_test, model_baseline, history_baseline, metadata, genre_columns
    except NameError:
        pass
    gc.collect()

Training Baseline Model...


I0000 00:00:1763479580.299948      19 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13942 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1763479580.300753      19 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13942 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


Epoch 1/10


I0000 00:00:1763479594.156464      73 service.cc:148] XLA service 0x7a05981897e0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1763479594.157385      73 service.cc:156]   StreamExecutor device (0): Tesla T4, Compute Capability 7.5
I0000 00:00:1763479594.157421      73 service.cc:156]   StreamExecutor device (1): Tesla T4, Compute Capability 7.5
I0000 00:00:1763479594.463470      73 cuda_dnn.cc:529] Loaded cuDNN version 90300
I0000 00:00:1763479596.778403      73 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


2442/2442 - 26s - 11ms/step - loss: 0.9824 - mae: 0.7656 - rmse: 0.9912 - val_loss: 0.7011 - val_mae: 0.6408 - val_rmse: 0.8373
Epoch 2/10
2442/2442 - 17s - 7ms/step - loss: 0.7640 - mae: 0.6761 - rmse: 0.8741 - val_loss: 0.6640 - val_mae: 0.6212 - val_rmse: 0.8149
Epoch 3/10
2442/2442 - 17s - 7ms/step - loss: 0.6683 - mae: 0.6268 - rmse: 0.8175 - val_loss: 0.6467 - val_mae: 0.6099 - val_rmse: 0.8042
Epoch 4/10
2442/2442 - 17s - 7ms/step - loss: 0.6205 - mae: 0.6004 - rmse: 0.7877 - val_loss: 0.6356 - val_mae: 0.6069 - val_rmse: 0.7972
Epoch 5/10
2442/2442 - 18s - 7ms/step - loss: 0.5957 - mae: 0.5871 - rmse: 0.7718 - val_loss: 0.6238 - val_mae: 0.5979 - val_rmse: 0.7898
Epoch 6/10
2442/2442 - 18s - 7ms/step - loss: 0.5776 - mae: 0.5780 - rmse: 0.7600 - val_loss: 0.6213 - val_mae: 0.5962 - val_rmse: 0.7882
Epoch 7/10
2442/2442 - 18s - 7ms/step - loss: 0.5631 - mae: 0.5705 - rmse: 0.7504 - val_loss: 0.6175 - val_mae: 0.5932 - val_rmse: 0.7858
Epoch 8/10
2442/2442 - 17s - 7ms/step - loss

In [5]:
try:
    print("Training Model V1...")
    train_df = pd.read_parquet(OUTPUT_DIR + 'train.parquet')
    val_df = pd.read_parquet(OUTPUT_DIR + 'val.parquet')
    test_df = pd.read_parquet(OUTPUT_DIR + 'test.parquet')
    metadata = joblib.load(OUTPUT_DIR + 'metadata.joblib')
    n_users = metadata['n_users']
    n_movies = metadata['n_movies']
    n_genres = metadata['n_genres']
    genre_columns = joblib.load(OUTPUT_DIR + 'genre_columns.joblib')

    def build_model_v1(n_users, n_movies, n_genres):
        user_input = Input(shape=(1,), name='user_input')
        movie_input = Input(shape=(1,), name='movie_input')
        genre_input = Input(shape=(n_genres,), name='genre_input')
        year_input = Input(shape=(1,), name='year_input')
        
        user_embedding = Embedding(n_users, 128, name='user_embedding')(user_input)
        user_vec = Flatten(name='flatten_user')(user_embedding)
        
        movie_embedding = Embedding(n_movies, 128, name='movie_embedding')(movie_input)
        movie_vec = Flatten(name='flatten_movie')(movie_embedding)
        
        genre_vec = Dense(16, activation='relu', name='genre_dense')(genre_input)
        
        concat = Concatenate()([user_vec, movie_vec, genre_vec, year_input])
        dense_1 = Dense(128, activation='relu')(concat)
        dropout_1 = Dropout(0.3)(dense_1)
        dense_2 = Dense(64, activation='relu')(dropout_1)
        dropout_2 = Dropout(0.3)(dense_2)
        output = Dense(1, activation='linear')(dropout_2)
        
        model = Model(inputs=[user_input, movie_input, genre_input, year_input], outputs=output)
        model.compile(optimizer=Adam(0.001), loss='mse', metrics=[tf.keras.metrics.RootMeanSquaredError(name='rmse'), tf.keras.metrics.MeanAbsoluteError(name='mae')])
        return model

    X_train = {'user_input': train_df['user_idx'], 'movie_input': train_df['movie_idx'], 'genre_input': train_df[genre_columns], 'year_input': train_df['year_scaled']}
    y_train = train_df['rating']
    X_val = {'user_input': val_df['user_idx'], 'movie_input': val_df['movie_idx'], 'genre_input': val_df[genre_columns], 'year_input': val_df['year_scaled']}
    y_val = val_df['rating']

    early_stopping = EarlyStopping(monitor='val_rmse', patience=2, mode='min', restore_best_weights=True)
    model_v1 = build_model_v1(n_users, n_movies, n_genres)
    history_v1 = model_v1.fit(X_train, y_train, batch_size=8192, epochs=20, verbose=2, validation_data=(X_val, y_val), callbacks=[early_stopping])

    model_v1.save(OUTPUT_DIR + 'model_v1.keras')
    pd.DataFrame(history_v1.history).to_csv(OUTPUT_DIR + 'history_v1.csv')

    X_test = {'user_input': test_df['user_idx'], 'movie_input': test_df['movie_idx'], 'genre_input': test_df[genre_columns], 'year_input': test_df['year_scaled']}
    y_test = test_df['rating']
    results_v1 = model_v1.evaluate(X_test, y_test, batch_size=8192)
    print(f"V1 RMSE: {results_v1[1]:.4f}")

except Exception as e:
    print(f"Error Cell 5: {e}")
finally:
    try:
        del train_df, val_df, test_df, X_train, y_train, X_val, y_val, X_test, y_test, model_v1, history_v1, metadata, genre_columns
    except NameError:
        pass
    gc.collect()

Training Model V1...
Epoch 1/20
2442/2442 - 33s - 14ms/step - loss: 0.9773 - mae: 0.7656 - rmse: 0.9886 - val_loss: 0.7011 - val_mae: 0.6433 - val_rmse: 0.8373
Epoch 2/20
2442/2442 - 25s - 10ms/step - loss: 0.7576 - mae: 0.6730 - rmse: 0.8704 - val_loss: 0.6643 - val_mae: 0.6226 - val_rmse: 0.8151
Epoch 3/20
2442/2442 - 25s - 10ms/step - loss: 0.6624 - mae: 0.6238 - rmse: 0.8138 - val_loss: 0.6488 - val_mae: 0.6106 - val_rmse: 0.8055
Epoch 4/20
2442/2442 - 26s - 11ms/step - loss: 0.6149 - mae: 0.5977 - rmse: 0.7841 - val_loss: 0.6315 - val_mae: 0.6018 - val_rmse: 0.7947
Epoch 5/20
2442/2442 - 25s - 10ms/step - loss: 0.5883 - mae: 0.5838 - rmse: 0.7670 - val_loss: 0.6216 - val_mae: 0.5964 - val_rmse: 0.7884
Epoch 6/20
2442/2442 - 25s - 10ms/step - loss: 0.5684 - mae: 0.5735 - rmse: 0.7539 - val_loss: 0.6179 - val_mae: 0.5946 - val_rmse: 0.7861
Epoch 7/20
2442/2442 - 25s - 10ms/step - loss: 0.5526 - mae: 0.5653 - rmse: 0.7433 - val_loss: 0.6146 - val_mae: 0.5930 - val_rmse: 0.7840
Epoch 

In [6]:
try:
    print("Training Model V2 (Wide & Deep)...")
    train_df = pd.read_parquet(OUTPUT_DIR + 'train.parquet')
    val_df = pd.read_parquet(OUTPUT_DIR + 'val.parquet')
    test_df = pd.read_parquet(OUTPUT_DIR + 'test.parquet')
    metadata = joblib.load(OUTPUT_DIR + 'metadata.joblib')
    n_users = metadata['n_users']
    n_movies = metadata['n_movies']
    n_genres = metadata['n_genres']
    genre_columns = joblib.load(OUTPUT_DIR + 'genre_columns.joblib')

    def build_model_v2(n_users, n_movies, n_genres):
        user_input = Input(shape=(1,), name='user_input')
        movie_input = Input(shape=(1,), name='movie_input')
        genre_input = Input(shape=(n_genres,), name='genre_input')
        year_input = Input(shape=(1,), name='year_input')
        
        user_embedding_deep = Embedding(n_users, 64, name='user_emb_deep')(user_input)
        user_vec_deep = Flatten(name='flat_user_deep')(user_embedding_deep)
        movie_embedding_deep = Embedding(n_movies, 64, name='movie_emb_deep')(movie_input)
        movie_vec_deep = Flatten(name='flat_movie_deep')(movie_embedding_deep)
        genre_vec_deep = Dense(16, activation='relu', name='genre_dense_deep')(genre_input)
        
        concat = Concatenate()([user_vec_deep, movie_vec_deep, genre_vec_deep, year_input])
        dense_1 = Dense(128, activation='relu', name='dense_1_deep')(concat)
        dropout_1 = Dropout(0.3, name='dropout_1_deep')(dense_1)
        dense_2 = Dense(64, activation='relu', name='dense_2_deep')(dropout_1)
        dropout_2 = Dropout(0.3, name='dropout_2_deep')(dense_2)
        deep_output = Dense(1, name='deep_output')(dropout_2)
        
        user_embedding_wide = Embedding(n_users, 64, name='user_emb_wide')(user_input)
        user_vec_wide = Flatten(name='flat_user_wide')(user_embedding_wide)
        movie_embedding_wide = Embedding(n_movies, 64, name='movie_emb_wide')(movie_input)
        movie_vec_wide = Flatten(name='flat_movie_wide')(movie_embedding_wide)
        wide_output = Dot(axes=1, name='wide_output')([user_vec_wide, movie_vec_wide])
        
        final_output = Add(name='add_wide_deep')([wide_output, deep_output])
        
        model = Model(inputs=[user_input, movie_input, genre_input, year_input], outputs=final_output)
        model.compile(optimizer=Adam(0.001), loss='mse', metrics=[tf.keras.metrics.RootMeanSquaredError(name='rmse'), tf.keras.metrics.MeanAbsoluteError(name='mae')])
        return model

    X_train = {'user_input': train_df['user_idx'], 'movie_input': train_df['movie_idx'], 'genre_input': train_df[genre_columns], 'year_input': train_df['year_scaled']}
    y_train = train_df['rating']
    X_val = {'user_input': val_df['user_idx'], 'movie_input': val_df['movie_idx'], 'genre_input': val_df[genre_columns], 'year_input': val_df['year_scaled']}
    y_val = val_df['rating']

    early_stopping = EarlyStopping(monitor='val_rmse', patience=2, mode='min', restore_best_weights=True)
    model_v2 = build_model_v2(n_users, n_movies, n_genres)
    history_v2 = model_v2.fit(X_train, y_train, batch_size=8192, epochs=20, verbose=2, validation_data=(X_val, y_val), callbacks=[early_stopping])

    model_v2.save(OUTPUT_DIR + 'model_v2.keras')
    pd.DataFrame(history_v2.history).to_csv(OUTPUT_DIR + 'history_v2.csv')

    X_test = {'user_input': test_df['user_idx'], 'movie_input': test_df['movie_idx'], 'genre_input': test_df[genre_columns], 'year_input': test_df['year_scaled']}
    y_test = test_df['rating']
    results_v2 = model_v2.evaluate(X_test, y_test, batch_size=8192)
    print(f"V2 RMSE: {results_v2[1]:.4f}")

except Exception as e:
    print(f"Error Cell 6: {e}")
finally:
    try:
        del train_df, val_df, test_df, X_train, y_train, X_val, y_val, X_test, y_test, model_v2, history_v2, metadata, genre_columns
    except NameError:
        pass
    gc.collect()

Training Model V2 (Wide & Deep)...
Epoch 1/20
2442/2442 - 34s - 14ms/step - loss: 1.0145 - mae: 0.7746 - rmse: 1.0072 - val_loss: 0.6640 - val_mae: 0.6254 - val_rmse: 0.8148
Epoch 2/20
2442/2442 - 26s - 11ms/step - loss: 0.6809 - mae: 0.6376 - rmse: 0.8252 - val_loss: 0.6101 - val_mae: 0.5937 - val_rmse: 0.7811
Epoch 3/20
2442/2442 - 26s - 11ms/step - loss: 0.5275 - mae: 0.5548 - rmse: 0.7263 - val_loss: 0.6078 - val_mae: 0.5924 - val_rmse: 0.7796
Epoch 4/20
2442/2442 - 26s - 10ms/step - loss: 0.4446 - mae: 0.5041 - rmse: 0.6668 - val_loss: 0.6190 - val_mae: 0.5967 - val_rmse: 0.7868
Epoch 5/20
2442/2442 - 25s - 10ms/step - loss: 0.4007 - mae: 0.4756 - rmse: 0.6330 - val_loss: 0.6329 - val_mae: 0.6025 - val_rmse: 0.7955
306/306 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 0.6076 - mae: 0.5923 - rmse: 0.7795
V2 RMSE: 0.7787


In [7]:
try:
    print("Training Model V2 Enhanced...")
    train_df = pd.read_parquet(OUTPUT_DIR + 'train.parquet')
    val_df = pd.read_parquet(OUTPUT_DIR + 'val.parquet')
    test_df = pd.read_parquet(OUTPUT_DIR + 'test.parquet')
    metadata = joblib.load(OUTPUT_DIR + 'metadata.joblib')
    n_users = metadata['n_users']
    n_movies = metadata['n_movies']
    n_genres = metadata['n_genres']
    genre_columns = joblib.load(OUTPUT_DIR + 'genre_columns.joblib')

    def build_model_v2_enhanced(n_users, n_movies, n_genres):
        user_input = Input(shape=(1,), name='user_input')
        movie_input = Input(shape=(1,), name='movie_input')
        genre_input = Input(shape=(n_genres,), name='genre_input')
        year_input = Input(shape=(1,), name='year_input')
        
        user_embedding_deep = Embedding(n_users, 64, name='user_emb_deep', embeddings_regularizer=l2(1e-5))(user_input)
        user_vec_deep = Flatten(name='flat_user_deep')(user_embedding_deep)
        movie_embedding_deep = Embedding(n_movies, 64, name='movie_emb_deep', embeddings_regularizer=l2(1e-5))(movie_input)
        movie_vec_deep = Flatten(name='flat_movie_deep')(movie_embedding_deep)
        genre_vec_deep = Dense(16, activation='relu', name='genre_dense_deep')(genre_input)
        
        concat = Concatenate()([user_vec_deep, movie_vec_deep, genre_vec_deep, year_input])
        dense_1 = Dense(128)(concat)
        norm_1 = LayerNormalization()(dense_1)
        act_1 = tf.keras.layers.Activation('relu')(norm_1)
        dense_2 = Dense(64)(act_1)
        norm_2 = LayerNormalization()(dense_2)
        act_2 = tf.keras.layers.Activation('relu')(norm_2)
        deep_output = Dense(1, name='deep_output')(act_2)
        
        user_embedding_wide = Embedding(n_users, 64, name='user_emb_wide', embeddings_regularizer=l2(1e-5))(user_input)
        user_vec_wide = Flatten(name='flat_user_wide')(user_embedding_wide)
        movie_embedding_wide = Embedding(n_movies, 64, name='movie_emb_wide', embeddings_regularizer=l2(1e-5))(movie_input)
        movie_vec_wide = Flatten(name='flat_movie_wide')(movie_embedding_wide)
        wide_output = Dot(axes=1, name='wide_output')([user_vec_wide, movie_vec_wide])
        
        final_output = Add(name='add_wide_deep')([wide_output, deep_output])
        
        model = Model(inputs=[user_input, movie_input, genre_input, year_input], outputs=final_output)
        model.compile(optimizer=Adam(0.001), loss='mse', metrics=[tf.keras.metrics.RootMeanSquaredError(name='rmse'), tf.keras.metrics.MeanAbsoluteError(name='mae')])
        return model

    X_train = {'user_input': train_df['user_idx'], 'movie_input': train_df['movie_idx'], 'genre_input': train_df[genre_columns], 'year_input': train_df['year_scaled']}
    y_train = train_df['rating']
    X_val = {'user_input': val_df['user_idx'], 'movie_input': val_df['movie_idx'], 'genre_input': val_df[genre_columns], 'year_input': val_df['year_scaled']}
    y_val = val_df['rating']

    early_stopping = EarlyStopping(monitor='val_rmse', patience=2, mode='min', restore_best_weights=True)
    model_v2_enhanced = build_model_v2_enhanced(n_users, n_movies, n_genres)
    history_v2_enhanced = model_v2_enhanced.fit(X_train, y_train, batch_size=8192, epochs=20, verbose=2, validation_data=(X_val, y_val), callbacks=[early_stopping])

    model_v2_enhanced.save(OUTPUT_DIR + 'model_v2_enhanced.keras')
    pd.DataFrame(history_v2_enhanced.history).to_csv(OUTPUT_DIR + 'history_v2_enhanced_history.csv')

    X_test = {'user_input': test_df['user_idx'], 'movie_input': test_df['movie_idx'], 'genre_input': test_df[genre_columns], 'year_input': test_df['year_scaled']}
    y_test = test_df['rating']
    results_v2_e = model_v2_enhanced.evaluate(X_test, y_test, batch_size=8192)
    print(f"V2 Enhanced RMSE: {results_v2_e[1]:.4f}")

except Exception as e:
    print(f"Error Cell 7: {e}")
finally:
    try:
        del train_df, val_df, test_df, X_train, y_train, X_val, y_val, X_test, y_test, model_v2_enhanced, history_v2_enhanced, metadata, genre_columns
    except NameError:
        pass
    gc.collect()

Training Model V2 Enhanced...
Epoch 1/20
2442/2442 - 36s - 15ms/step - loss: 0.8017 - mae: 0.6567 - rmse: 0.8638 - val_loss: 0.7379 - val_mae: 0.6253 - val_rmse: 0.8222
Epoch 2/20
2442/2442 - 26s - 11ms/step - loss: 0.7161 - mae: 0.6114 - rmse: 0.8036 - val_loss: 0.7123 - val_mae: 0.6086 - val_rmse: 0.7999
Epoch 3/20
2442/2442 - 26s - 11ms/step - loss: 0.6878 - mae: 0.5931 - rmse: 0.7804 - val_loss: 0.6991 - val_mae: 0.5984 - val_rmse: 0.7884
Epoch 4/20
2442/2442 - 26s - 11ms/step - loss: 0.6705 - mae: 0.5813 - rmse: 0.7653 - val_loss: 0.6949 - val_mae: 0.5925 - val_rmse: 0.7824
Epoch 5/20
2442/2442 - 26s - 11ms/step - loss: 0.6605 - mae: 0.5737 - rmse: 0.7552 - val_loss: 0.6948 - val_mae: 0.5889 - val_rmse: 0.7793
Epoch 6/20
2442/2442 - 26s - 11ms/step - loss: 0.6539 - mae: 0.5681 - rmse: 0.7477 - val_loss: 0.6979 - val_mae: 0.5911 - val_rmse: 0.7785
Epoch 7/20
2442/2442 - 26s - 11ms/step - loss: 0.6492 - mae: 0.5638 - rmse: 0.7420 - val_loss: 0.7008 - val_mae: 0.5910 - val_rmse: 0.77

In [8]:
try:
    print("Training Model DCN...")
    train_df = pd.read_parquet(OUTPUT_DIR + 'train.parquet')
    val_df = pd.read_parquet(OUTPUT_DIR + 'val.parquet')
    test_df = pd.read_parquet(OUTPUT_DIR + 'test.parquet')
    metadata = joblib.load(OUTPUT_DIR + 'metadata.joblib')
    n_users = metadata['n_users']
    n_movies = metadata['n_movies']
    n_genres = metadata['n_genres']
    genre_columns = joblib.load(OUTPUT_DIR + 'genre_columns.joblib')

    class CrossLayer(tf.keras.layers.Layer):
        def __init__(self, kernel_regularizer=None, **kwargs):
            super(CrossLayer, self).__init__(**kwargs)
            self.kernel_regularizer = kernel_regularizer
        def build(self, input_shape):
            dim = input_shape[0][-1]
            self.W = self.add_weight(shape=(dim, 1), initializer='glorot_uniform', regularizer=self.kernel_regularizer, name='cross_kernel')
            self.b = self.add_weight(shape=(dim,), initializer='zeros', name='cross_bias')
        def call(self, inputs):
            x_0, x_l = inputs
            x_l_dot_W = tf.tensordot(x_l, self.W, axes=[1, 0])
            cross_term = x_0 * x_l_dot_W
            return cross_term + self.b + x_l
        def get_config(self):
            config = super(CrossLayer, self).get_config()
            config.update({'kernel_regularizer': self.kernel_regularizer})
            return config

    def build_dcn_model(n_users, n_movies, n_genres, num_cross_layers=3):
        user_input = Input(shape=(1,), name='user_input')
        movie_input = Input(shape=(1,), name='movie_input')
        genre_input = Input(shape=(n_genres,), name='genre_input')
        year_input = Input(shape=(1,), name='year_input')
        
        user_embedding = Embedding(n_users, 64, name='user_embedding', embeddings_regularizer=l2(1e-5))(user_input)
        user_vec = Flatten(name='flatten_user')(user_embedding)
        movie_embedding = Embedding(n_movies, 64, name='movie_embedding', embeddings_regularizer=l2(1e-5))(movie_input)
        movie_vec = Flatten(name='flatten_movie')(movie_embedding)
        genre_vec = Dense(16, activation='relu', name='genre_dense')(genre_input)
        
        x_0 = Concatenate()([user_vec, movie_vec, genre_vec, year_input])
        deep = Dense(128)(x_0)
        deep = LayerNormalization()(deep)
        deep = tf.keras.layers.Activation('relu')(deep)
        deep = Dense(64)(deep)
        deep = LayerNormalization()(deep)
        deep_output = tf.keras.layers.Activation('relu')(deep)
        
        x_l = x_0
        for _ in range(num_cross_layers):
            x_l = CrossLayer()([x_0, x_l]) 
        cross_output = x_l
        
        concat_output = Concatenate()([cross_output, deep_output])
        final_output = Dense(1, activation='linear', name='output')(concat_output)
        
        model = Model(inputs=[user_input, movie_input, genre_input, year_input], outputs=final_output)
        model.compile(optimizer=Adam(0.001), loss='mse', metrics=[tf.keras.metrics.RootMeanSquaredError(name='rmse'), tf.keras.metrics.MeanAbsoluteError(name='mae')])
        return model

    X_train = {'user_input': train_df['user_idx'], 'movie_input': train_df['movie_idx'], 'genre_input': train_df[genre_columns], 'year_input': train_df['year_scaled']}
    y_train = train_df['rating']
    X_val = {'user_input': val_df['user_idx'], 'movie_input': val_df['movie_idx'], 'genre_input': val_df[genre_columns], 'year_input': val_df['year_scaled']}
    y_val = val_df['rating']

    early_stopping = EarlyStopping(monitor='val_rmse', patience=2, mode='min', restore_best_weights=True)
    model_dcn = build_dcn_model(n_users, n_movies, n_genres)
    history_dcn = model_dcn.fit(X_train, y_train, batch_size=8192, epochs=20, verbose=2, validation_data=(X_val, y_val), callbacks=[early_stopping])

    model_dcn.save(OUTPUT_DIR + 'model_dcn.keras')
    pd.DataFrame(history_dcn.history).to_csv(OUTPUT_DIR + 'history_dcn.csv')

    X_test = {'user_input': test_df['user_idx'], 'movie_input': test_df['movie_idx'], 'genre_input': test_df[genre_columns], 'year_input': test_df['year_scaled']}
    y_test = test_df['rating']
    results_dcn = model_dcn.evaluate(X_test, y_test, batch_size=8192)
    print(f"DCN RMSE: {results_dcn[1]:.4f}")

except Exception as e:
    print(f"Error Cell 8: {e}")
finally:
    try:
        del train_df, val_df, test_df, X_train, y_train, X_val, y_val, X_test, y_test, model_dcn, history_dcn, metadata, genre_columns
    except NameError:
        pass
    gc.collect()

Training Model DCN...
Epoch 1/20
2442/2442 - 30s - 12ms/step - loss: 0.7967 - mae: 0.6560 - rmse: 0.8630 - val_loss: 0.7343 - val_mae: 0.6245 - val_rmse: 0.8198
Epoch 2/20
2442/2442 - 19s - 8ms/step - loss: 0.7138 - mae: 0.6110 - rmse: 0.8028 - val_loss: 0.7118 - val_mae: 0.6064 - val_rmse: 0.8000
Epoch 3/20
2442/2442 - 19s - 8ms/step - loss: 0.6887 - mae: 0.5939 - rmse: 0.7812 - val_loss: 0.7007 - val_mae: 0.5976 - val_rmse: 0.7893
Epoch 4/20
2442/2442 - 19s - 8ms/step - loss: 0.6721 - mae: 0.5827 - rmse: 0.7669 - val_loss: 0.6959 - val_mae: 0.5926 - val_rmse: 0.7836
Epoch 5/20
2442/2442 - 19s - 8ms/step - loss: 0.6614 - mae: 0.5750 - rmse: 0.7569 - val_loss: 0.6948 - val_mae: 0.5908 - val_rmse: 0.7804
Epoch 6/20
2442/2442 - 19s - 8ms/step - loss: 0.6545 - mae: 0.5694 - rmse: 0.7495 - val_loss: 0.6967 - val_mae: 0.5896 - val_rmse: 0.7788
Epoch 7/20
2442/2442 - 19s - 8ms/step - loss: 0.6497 - mae: 0.5651 - rmse: 0.7437 - val_loss: 0.6988 - val_mae: 0.5888 - val_rmse: 0.7778
Epoch 8/20


In [9]:
try:
    print("Training Model NCF++...")
    train_df = pd.read_parquet(OUTPUT_DIR + 'train.parquet')
    val_df = pd.read_parquet(OUTPUT_DIR + 'val.parquet')
    test_df = pd.read_parquet(OUTPUT_DIR + 'test.parquet')
    metadata = joblib.load(OUTPUT_DIR + 'metadata.joblib')
    n_users = metadata['n_users']
    n_movies = metadata['n_movies']
    n_genres = metadata['n_genres']
    genre_columns = joblib.load(OUTPUT_DIR + 'genre_columns.joblib')

    def build_ncf_model(n_users, n_movies, n_genres, mf_dim=64, mlp_dims=[128, 64, 32]):
        user_input = Input(shape=(1,), name='user_input')
        movie_input = Input(shape=(1,), name='movie_input')
        genre_input = Input(shape=(n_genres,), name='genre_input')
        year_input = Input(shape=(1,), name='year_input')
        
        user_embedding_gmf = Embedding(n_users, mf_dim, name='user_emb_gmf', embeddings_regularizer=l2(1e-5))(user_input)
        user_vec_gmf = Flatten()(user_embedding_gmf)
        movie_embedding_gmf = Embedding(n_movies, mf_dim, name='movie_emb_gmf', embeddings_regularizer=l2(1e-5))(movie_input)
        movie_vec_gmf = Flatten()(movie_embedding_gmf)
        gmf_output = tf.keras.layers.Multiply()([user_vec_gmf, movie_vec_gmf])
        
        user_embedding_mlp = Embedding(n_users, mf_dim, name='user_emb_mlp', embeddings_regularizer=l2(1e-5))(user_input)
        user_vec_mlp = Flatten()(user_embedding_mlp)
        movie_embedding_mlp = Embedding(n_movies, mf_dim, name='movie_emb_mlp', embeddings_regularizer=l2(1e-5))(movie_input)
        movie_vec_mlp = Flatten()(movie_embedding_mlp)
        genre_vec = Dense(16, activation='relu', name='genre_dense')(genre_input)
        
        mlp_input = Concatenate()([user_vec_mlp, movie_vec_mlp, genre_vec, year_input])
        mlp_layer = mlp_input
        for dim in mlp_dims:
            mlp_layer = Dense(dim)(mlp_layer)
            mlp_layer = LayerNormalization()(mlp_layer)
            mlp_layer = tf.keras.layers.Activation('relu')(mlp_layer)
        mlp_output = mlp_layer
        
        concat_output = Concatenate()([gmf_output, mlp_output])
        final_output = Dense(1, activation='linear', name='output')(concat_output)
        
        model = Model(inputs=[user_input, movie_input, genre_input, year_input], outputs=final_output)
        model.compile(optimizer=Adam(0.001), loss='mse', metrics=[tf.keras.metrics.RootMeanSquaredError(name='rmse'), tf.keras.metrics.MeanAbsoluteError(name='mae')])
        return model

    X_train = {'user_input': train_df['user_idx'], 'movie_input': train_df['movie_idx'], 'genre_input': train_df[genre_columns], 'year_input': train_df['year_scaled']}
    y_train = train_df['rating']
    X_val = {'user_input': val_df['user_idx'], 'movie_input': val_df['movie_idx'], 'genre_input': val_df[genre_columns], 'year_input': val_df['year_scaled']}
    y_val = val_df['rating']

    early_stopping = EarlyStopping(monitor='val_rmse', patience=2, mode='min', restore_best_weights=True)
    model_ncf = build_ncf_model(n_users, n_movies, n_genres)
    history_ncf = model_ncf.fit(X_train, y_train, batch_size=8192, epochs=20, verbose=2, validation_data=(X_val, y_val), callbacks=[early_stopping])

    model_ncf.save(OUTPUT_DIR + 'model_ncf.keras')
    pd.DataFrame(history_ncf.history).to_csv(OUTPUT_DIR + 'history_ncf.csv')

    X_test = {'user_input': test_df['user_idx'], 'movie_input': test_df['movie_idx'], 'genre_input': test_df[genre_columns], 'year_input': test_df['year_scaled']}
    y_test = test_df['rating']
    results_ncf = model_ncf.evaluate(X_test, y_test, batch_size=8192)
    print(f"NCF RMSE: {results_ncf[1]:.4f}")

except Exception as e:
    print(f"Error Cell 9: {e}")
finally:
    try:
        del train_df, val_df, test_df, X_train, y_train, X_val, y_val, X_test, y_test, model_ncf, history_ncf, metadata, genre_columns
    except NameError:
        pass
    gc.collect()

Training Model NCF++...
Epoch 1/20
2442/2442 - 38s - 15ms/step - loss: 0.8192 - mae: 0.6663 - rmse: 0.8760 - val_loss: 0.7328 - val_mae: 0.6244 - val_rmse: 0.8196
Epoch 2/20
2442/2442 - 27s - 11ms/step - loss: 0.7100 - mae: 0.6100 - rmse: 0.8016 - val_loss: 0.7065 - val_mae: 0.6052 - val_rmse: 0.7988
Epoch 3/20
2442/2442 - 27s - 11ms/step - loss: 0.6835 - mae: 0.5927 - rmse: 0.7797 - val_loss: 0.6957 - val_mae: 0.5996 - val_rmse: 0.7879
Epoch 4/20
2442/2442 - 27s - 11ms/step - loss: 0.6688 - mae: 0.5820 - rmse: 0.7660 - val_loss: 0.6932 - val_mae: 0.5934 - val_rmse: 0.7829
Epoch 5/20
2442/2442 - 27s - 11ms/step - loss: 0.6597 - mae: 0.5746 - rmse: 0.7564 - val_loss: 0.6935 - val_mae: 0.5916 - val_rmse: 0.7799
Epoch 6/20
2442/2442 - 27s - 11ms/step - loss: 0.6536 - mae: 0.5691 - rmse: 0.7490 - val_loss: 0.6947 - val_mae: 0.5881 - val_rmse: 0.7779
Epoch 7/20
2442/2442 - 27s - 11ms/step - loss: 0.6491 - mae: 0.5648 - rmse: 0.7432 - val_loss: 0.6975 - val_mae: 0.5873 - val_rmse: 0.7772
Epo

In [10]:
try:
    print("Training Model V3 (Genome)...")
    class GenomeDataGenerator(Sequence):
        def __init__(self, df, genre_cols, genome_dense_matrix, batch_size, **kwargs):
            super().__init__(**kwargs) 
            self.df = df
            self.genre_cols = genre_cols
            self.genome_dense_matrix = genome_dense_matrix 
            self.batch_size = batch_size
            self.indices = self.df.index.tolist()
        def __len__(self):
            return int(np.floor(len(self.indices) / self.batch_size))
        def __getitem__(self, index):
            start = index * self.batch_size
            end = (index + 1) * self.batch_size
            batch_indices = self.indices[start:end]
            batch_df = self.df.loc[batch_indices]
            batch_movie_idx = batch_df['movie_idx'].values
            batch_genome_data = self.genome_dense_matrix[batch_movie_idx]
            X_batch = {
                'user_input': batch_df['user_idx'].values,
                'movie_input': batch_df['movie_idx'].values,
                'genre_input': batch_df[self.genre_cols].values,
                'year_input': batch_df['year_scaled'].values,
                'genome_input': batch_genome_data
            }
            y_batch = batch_df['rating'].values
            return X_batch, y_batch

    train_df = pd.read_parquet(OUTPUT_DIR + 'train.parquet')
    val_df = pd.read_parquet(OUTPUT_DIR + 'val.parquet')
    test_df = pd.read_parquet(OUTPUT_DIR + 'test.parquet')
    metadata = joblib.load(OUTPUT_DIR + 'metadata.joblib')
    n_users = metadata['n_users']
    n_movies = metadata['n_movies']
    n_genres = metadata['n_genres']
    n_tags = metadata['n_tags'] 
    genre_columns = joblib.load(OUTPUT_DIR + 'genre_columns.joblib')
    
    genome_sparse_matrix = sp.load_npz(OUTPUT_DIR + 'genome_sparse_matrix.npz')
    genome_dense_matrix = genome_sparse_matrix.toarray().astype(np.float32)
    del genome_sparse_matrix
    gc.collect()

    def build_model_v3(n_users, n_movies, n_genres, n_genome_tags):
        user_input = Input(shape=(1,), name='user_input')
        movie_input = Input(shape=(1,), name='movie_input')
        genre_input = Input(shape=(n_genres,), name='genre_input')
        year_input = Input(shape=(1,), name='year_input')
        genome_input = Input(shape=(n_genome_tags,), name='genome_input')
        
        user_embedding = Embedding(n_users, 64, embeddings_regularizer=l2(1e-5))(user_input)
        user_vec = Flatten()(user_embedding)
        movie_embedding = Embedding(n_movies, 64, embeddings_regularizer=l2(1e-5))(movie_input)
        movie_vec = Flatten()(movie_embedding)
        genre_vec = Dense(16, activation='relu')(genre_input)
        genome_vec = Dense(64, activation='relu')(genome_input)
        
        concat = Concatenate()([user_vec, movie_vec, genre_vec, year_input, genome_vec])
        dense_1 = Dense(128)(concat)
        norm_1 = LayerNormalization()(dense_1)
        act_1 = tf.keras.layers.Activation('relu')(norm_1)
        dense_2 = Dense(64)(act_1)
        norm_2 = LayerNormalization()(dense_2)
        act_2 = tf.keras.layers.Activation('relu')(norm_2)
        output = Dense(1, activation='linear')(act_2)
        
        model = Model(inputs=[user_input, movie_input, genre_input, year_input, genome_input], outputs=output)
        model.compile(optimizer=Adam(0.001), loss='mse', metrics=[tf.keras.metrics.RootMeanSquaredError(name='rmse'), tf.keras.metrics.MeanAbsoluteError(name='mae')])
        return model

    BATCH_SIZE = 4096 
    train_gen = GenomeDataGenerator(train_df, genre_columns, genome_dense_matrix, BATCH_SIZE)
    val_gen = GenomeDataGenerator(val_df, genre_columns, genome_dense_matrix, BATCH_SIZE)

    early_stopping = EarlyStopping(monitor='val_rmse', patience=2, mode='min', restore_best_weights=True)
    reduce_lr = ReduceLROnPlateau(monitor='val_rmse', factor=0.2, patience=1, min_lr=0.0001)
    
    model_v3 = build_model_v3(n_users, n_movies, n_genres, n_tags)
    history_v3 = model_v3.fit(train_gen, epochs=20, verbose=1, validation_data=val_gen, callbacks=[early_stopping, reduce_lr])

    model_v3.save(OUTPUT_DIR + 'model_v3.keras')
    pd.DataFrame(history_v3.history).to_csv(OUTPUT_DIR + 'history_v3.csv')

    test_gen = GenomeDataGenerator(test_df, genre_columns, genome_dense_matrix, BATCH_SIZE)
    results_v3 = model_v3.evaluate(test_gen)
    print(f"V3 RMSE: {results_v3[1]:.4f}")

except Exception as e:
    print(f"Error Cell 10: {e}")
finally:
    try:
        del train_df, val_df, test_df, metadata, genre_columns, genome_dense_matrix, model_v3, history_v3, train_gen, val_gen, test_gen
    except NameError:
        pass
    gc.collect()

Training Model V3 (Genome)...
Epoch 1/20
4882/4882 ━━━━━━━━━━━━━━━━━━━━ 70s 13ms/step - loss: 0.8284 - mae: 0.6752 - rmse: 0.8841 - val_loss: 0.7296 - val_mae: 0.6242 - val_rmse: 0.8199 - learning_rate: 0.0010
Epoch 2/20
4882/4882 ━━━━━━━━━━━━━━━━━━━━ 65s 13ms/step - loss: 0.7113 - mae: 0.6149 - rmse: 0.8078 - val_loss: 0.6991 - val_mae: 0.6063 - val_rmse: 0.7997 - learning_rate: 0.0010
Epoch 3/20
4882/4882 ━━━━━━━━━━━━━━━━━━━━ 84s 17ms/step - loss: 0.6786 - mae: 0.5965 - rmse: 0.7845 - val_loss: 0.6909 - val_mae: 0.6030 - val_rmse: 0.7917 - learning_rate: 0.0010
Epoch 4/20
4882/4882 ━━━━━━━━━━━━━━━━━━━━ 65s 13ms/step - loss: 0.6659 - mae: 0.5876 - rmse: 0.7732 - val_loss: 0.6862 - val_mae: 0.5979 - val_rmse: 0.7868 - learning_rate: 0.0010
Epoch 5/20
4882/4882 ━━━━━━━━━━━━━━━━━━━━ 67s 14ms/step - loss: 0.6577 - mae: 0.5818 - rmse: 0.7658 - val_loss: 0.6830 - val_mae: 0.5947 - val_rmse: 0.7833 - learning_rate: 0.0010
Epoch 6/20
4882/4882 ━━━━━━━━━━━━━━━━━━━━ 71s 15ms/step - loss: 0.6531

In [11]:
try:
    print("Processing User Tags...")
    user_encoder = joblib.load(OUTPUT_DIR + 'user_encoder.joblib')
    movie_encoder = joblib.load(OUTPUT_DIR + 'movie_encoder.joblib')
    metadata = joblib.load(OUTPUT_DIR + 'metadata.joblib')
    n_users = metadata['n_users']
    n_movies = metadata['n_movies']

    tags_df = pd.read_csv(BASE_PATH + 'tags.csv')
    tags_df['tag'] = tags_df['tag'].str.lower().str.replace(r'[^a-z0-9 ]', '', regex=True).str.strip()
    tags_df = tags_df.dropna(subset=['tag'])
    tags_df = tags_df[tags_df['tag'] != '']

    user_id_map = dict(zip(user_encoder.classes_, user_encoder.transform(user_encoder.classes_)))
    movie_id_map = dict(zip(movie_encoder.classes_, movie_encoder.transform(movie_encoder.classes_)))
    tags_df['user_idx'] = tags_df['userId'].map(user_id_map)
    tags_df['movie_idx'] = tags_df['movieId'].map(movie_id_map)
    tags_df = tags_df.dropna(subset=['user_idx', 'movie_idx'])
    tags_df['user_idx'] = tags_df['user_idx'].astype(int)
    tags_df['movie_idx'] = tags_df['movie_idx'].astype(int)

    user_docs = tags_df.groupby('user_idx')['tag'].apply(lambda x: ' '.join(x))
    user_tag_vectorizer = TfidfVectorizer(max_features=500) 
    user_tag_matrix_sparse = user_tag_vectorizer.fit_transform(user_docs)
    user_tag_matrix_full = sp.csr_matrix((n_users, 500))
    user_tag_matrix_full[user_docs.index] = user_tag_matrix_sparse
    user_tag_dense_matrix = user_tag_matrix_full.toarray().astype(np.float32)
    
    movie_docs = tags_df.groupby('movie_idx')['tag'].apply(lambda x: ' '.join(x))
    movie_tag_matrix_sparse = user_tag_vectorizer.transform(movie_docs)
    movie_tag_matrix_full = sp.csr_matrix((n_movies, 500))
    movie_tag_matrix_full[movie_docs.index] = movie_tag_matrix_sparse
    movie_tag_dense_matrix = movie_tag_matrix_full.toarray().astype(np.float32)
    
    np.save(OUTPUT_DIR + 'user_tag_dense_matrix.npy', user_tag_dense_matrix)
    np.save(OUTPUT_DIR + 'movie_tag_dense_matrix.npy', movie_tag_dense_matrix)
    joblib.dump(user_tag_vectorizer, OUTPUT_DIR + 'tag_vectorizer.joblib')

    metadata['n_user_tags'] = 500
    joblib.dump(metadata, OUTPUT_DIR + 'metadata.joblib')
    print("Cell 11 Completed")

except Exception as e:
    print(f"Error Cell 11: {e}")
finally:
    try:
        del tags_df, user_docs, movie_docs, user_tag_matrix_sparse, movie_tag_matrix_sparse, user_tag_matrix_full, movie_tag_matrix_full, user_tag_dense_matrix, movie_tag_dense_matrix, user_encoder, movie_encoder, metadata
    except NameError:
        pass
    gc.collect()

Processing User Tags...


/usr/local/lib/python3.11/dist-packages/scipy/sparse/_index.py:201: SparseEfficiencyWarning: Changing the sparsity structure of a csr_matrix is expensive. lil and dok are more efficient.
  self._set_arrayXarray_sparse(i, j, x)


Cell 11 Completed


In [12]:
try:
    print("Training Model V4 (Tags)...")
    class TagDataGenerator(Sequence):
        def __init__(self, df, genre_cols, user_tag_dense_matrix, movie_tag_dense_matrix, batch_size, **kwargs):
            super().__init__(**kwargs) 
            self.df = df
            self.genre_cols = genre_cols
            self.user_tag_dense_matrix = user_tag_dense_matrix
            self.movie_tag_dense_matrix = movie_tag_dense_matrix
            self.batch_size = batch_size
            self.indices = self.df.index.tolist()
        def __len__(self):
            return int(np.floor(len(self.indices) / self.batch_size))
        def __getitem__(self, index):
            start = index * self.batch_size
            end = (index + 1) * self.batch_size
            batch_indices = self.indices[start:end]
            batch_df = self.df.loc[batch_indices]
            batch_user_idx = batch_df['user_idx'].values
            batch_movie_idx = batch_df['movie_idx'].values
            batch_user_tags = self.user_tag_dense_matrix[batch_user_idx]
            batch_movie_tags = self.movie_tag_dense_matrix[batch_movie_idx]
            X_batch = {
                'user_input': batch_user_idx,
                'movie_input': batch_movie_idx,
                'genre_input': batch_df[self.genre_cols].values,
                'year_input': batch_df['year_scaled'].values,
                'user_tag_input': batch_user_tags,
                'movie_tag_input': batch_movie_tags
            }
            y_batch = batch_df['rating'].values
            return X_batch, y_batch

    class CrossLayer(tf.keras.layers.Layer):
        def __init__(self, kernel_regularizer=None, **kwargs):
            super(CrossLayer, self).__init__(**kwargs)
            self.kernel_regularizer = kernel_regularizer
        def build(self, input_shape):
            dim = input_shape[0][-1]
            self.W = self.add_weight(shape=(dim, 1), initializer='glorot_uniform', regularizer=self.kernel_regularizer, name='cross_kernel')
            self.b = self.add_weight(shape=(dim,), initializer='zeros', name='cross_bias')
        def call(self, inputs):
            x_0, x_l = inputs
            x_l_dot_W = tf.tensordot(x_l, self.W, axes=[1, 0])
            cross_term = x_0 * x_l_dot_W
            return cross_term + self.b + x_l
        def get_config(self):
            config = super(CrossLayer, self).get_config()
            config.update({'kernel_regularizer': self.kernel_regularizer})
            return config

    train_df = pd.read_parquet(OUTPUT_DIR + 'train.parquet')
    val_df = pd.read_parquet(OUTPUT_DIR + 'val.parquet')
    test_df = pd.read_parquet(OUTPUT_DIR + 'test.parquet')
    metadata = joblib.load(OUTPUT_DIR + 'metadata.joblib')
    n_users = metadata['n_users']
    n_movies = metadata['n_movies']
    n_genres = metadata['n_genres']
    n_user_tags = metadata['n_user_tags']
    genre_columns = joblib.load(OUTPUT_DIR + 'genre_columns.joblib')
    
    user_tag_dense_matrix = np.load(OUTPUT_DIR + 'user_tag_dense_matrix.npy')
    movie_tag_dense_matrix = np.load(OUTPUT_DIR + 'movie_tag_dense_matrix.npy')

    def build_model_v4_tags(n_users, n_movies, n_genres, n_user_tags, num_cross_layers=3):
        user_input = Input(shape=(1,), name='user_input')
        movie_input = Input(shape=(1,), name='movie_input')
        genre_input = Input(shape=(n_genres,), name='genre_input')
        year_input = Input(shape=(1,), name='year_input')
        user_tag_input = Input(shape=(n_user_tags,), name='user_tag_input')
        movie_tag_input = Input(shape=(n_user_tags,), name='movie_tag_input')
        
        user_embedding = Embedding(n_users, 64, name='user_embedding', embeddings_regularizer=l2(1e-5))(user_input)
        user_vec = Flatten(name='flatten_user')(user_embedding)
        movie_embedding = Embedding(n_movies, 64, name='movie_embedding', embeddings_regularizer=l2(1e-5))(movie_input)
        movie_vec = Flatten(name='flatten_movie')(movie_embedding)
        genre_vec = Dense(16, activation='relu', name='genre_dense')(genre_input)
        user_tag_vec = Dense(32, activation='relu', name='user_tag_dense')(user_tag_input)
        movie_tag_vec = Dense(32, activation='relu', name='movie_tag_dense')(movie_tag_input)
        
        x_0 = Concatenate()([user_vec, movie_vec, genre_vec, year_input, user_tag_vec, movie_tag_vec]) 
        deep = Dense(128)(x_0)
        deep = LayerNormalization()(deep)
        deep = tf.keras.layers.Activation('relu')(deep)
        deep = Dense(64)(deep)
        deep = LayerNormalization()(deep)
        deep_output = tf.keras.layers.Activation('relu')(deep)
        
        x_l = x_0
        for _ in range(num_cross_layers):
            x_l = CrossLayer()([x_0, x_l])
        cross_output = x_l
        
        concat_output = Concatenate()([cross_output, deep_output])
        final_output = Dense(1, activation='linear', name='output')(concat_output)
        
        model = Model(inputs=[user_input, movie_input, genre_input, year_input, user_tag_input, movie_tag_input], outputs=final_output)
        model.compile(optimizer=Adam(0.001), loss='mse', metrics=[tf.keras.metrics.RootMeanSquaredError(name='rmse'), tf.keras.metrics.MeanAbsoluteError(name='mae')])
        return model

    BATCH_SIZE = 4096
    train_gen_tag = TagDataGenerator(train_df, genre_columns, user_tag_dense_matrix, movie_tag_dense_matrix, BATCH_SIZE)
    val_gen_tag = TagDataGenerator(val_df, genre_columns, user_tag_dense_matrix, movie_tag_dense_matrix, BATCH_SIZE)

    early_stopping = EarlyStopping(monitor='val_rmse', patience=2, mode='min', restore_best_weights=True)
    reduce_lr = ReduceLROnPlateau(monitor='val_rmse', factor=0.2, patience=1, min_lr=0.0001)
    
    model_v4 = build_model_v4_tags(n_users, n_movies, n_genres, n_user_tags)
    history_v4 = model_v4.fit(train_gen_tag, epochs=20, verbose=1, validation_data=val_gen_tag, callbacks=[early_stopping, reduce_lr])

    model_v4.save(OUTPUT_DIR + 'model_v4.keras')
    pd.DataFrame(history_v4.history).to_csv(OUTPUT_DIR + 'history_v4.csv')

    test_gen_tag = TagDataGenerator(test_df, genre_columns, user_tag_dense_matrix, movie_tag_dense_matrix, BATCH_SIZE)
    results_v4 = model_v4.evaluate(test_gen_tag)
    print(f"V4 RMSE: {results_v4[1]:.4f}")

except Exception as e:
    print(f"Error Cell 12: {e}")
finally:
    try:
        del train_df, val_df, test_df, metadata, genre_columns, user_tag_dense_matrix, movie_tag_dense_matrix, model_v4, history_v4, train_gen_tag, val_gen_tag, test_gen_tag
    except NameError:
        pass
    gc.collect()

Training Model V4 (Tags)...
Epoch 1/20
4882/4882 ━━━━━━━━━━━━━━━━━━━━ 77s 14ms/step - loss: 0.8989 - mae: 0.6947 - rmse: 0.9126 - val_loss: 0.7364 - val_mae: 0.6244 - val_rmse: 0.8188 - learning_rate: 0.0010
Epoch 2/20
4882/4882 ━━━━━━━━━━━━━━━━━━━━ 68s 14ms/step - loss: 0.7140 - mae: 0.6105 - rmse: 0.8024 - val_loss: 0.7078 - val_mae: 0.6079 - val_rmse: 0.7982 - learning_rate: 0.0010
Epoch 3/20
4882/4882 ━━━━━━━━━━━━━━━━━━━━ 67s 14ms/step - loss: 0.6828 - mae: 0.5925 - rmse: 0.7796 - val_loss: 0.6984 - val_mae: 0.5979 - val_rmse: 0.7904 - learning_rate: 0.0010
Epoch 4/20
4882/4882 ━━━━━━━━━━━━━━━━━━━━ 66s 13ms/step - loss: 0.6686 - mae: 0.5832 - rmse: 0.7678 - val_loss: 0.6939 - val_mae: 0.5960 - val_rmse: 0.7856 - learning_rate: 0.0010
Epoch 5/20
4882/4882 ━━━━━━━━━━━━━━━━━━━━ 72s 15ms/step - loss: 0.6604 - mae: 0.5775 - rmse: 0.7604 - val_loss: 0.6939 - val_mae: 0.5969 - val_rmse: 0.7841 - learning_rate: 0.0010
Epoch 6/20
4882/4882 ━━━━━━━━━━━━━━━━━━━━ 72s 15ms/step - loss: 0.6555 -

In [13]:
try:
    print("Loading Test Data...")
    import pandas as pd
    import numpy as np
    import tensorflow as tf
    from tensorflow.keras.models import load_model
    from tensorflow.keras.layers import Layer
    import scipy.sparse as sp
    import joblib
    import os
    import gc

    class CrossLayer(tf.keras.layers.Layer):
        def __init__(self, kernel_regularizer=None, **kwargs):
            super(CrossLayer, self).__init__(**kwargs)
            self.kernel_regularizer = kernel_regularizer
        def build(self, input_shape):
            dim = input_shape[0][-1]
            self.W = self.add_weight(shape=(dim, 1), initializer='glorot_uniform', regularizer=self.kernel_regularizer, name='cross_kernel')
            self.b = self.add_weight(shape=(dim,), initializer='zeros', name='cross_bias')
        def call(self, inputs):
            x_0, x_l = inputs
            x_l_dot_W = tf.tensordot(x_l, self.W, axes=[1, 0])
            cross_term = x_0 * x_l_dot_W
            return cross_term + self.b + x_l
        def get_config(self):
            config = super(CrossLayer, self).get_config()
            config.update({'kernel_regularizer': self.kernel_regularizer})
            return config

    OUTPUT_DIR = '/kaggle/working/'
    test_df = pd.read_parquet(OUTPUT_DIR + 'test.parquet')
    y_test = test_df['rating'].values
    np.save(OUTPUT_DIR + 'y_test.npy', y_test)
    print("Cell 13 Completed")

except Exception as e:
    print(f"Error Cell 13: {e}")

Loading Test Data...
Cell 13 Completed


In [14]:
try:
    print("Predicting 4-Input Models...")
    try:
        _ = test_df
    except NameError:
        test_df = pd.read_parquet(OUTPUT_DIR + 'test.parquet')
        
    genre_columns = joblib.load(OUTPUT_DIR + 'genre_columns.joblib')
    X_test_4_input = {
        'user_input': test_df['user_idx'],
        'movie_input': test_df['movie_idx'],
        'genre_input': test_df[genre_columns].values,
        'year_input': test_df['year_scaled']
    }
    
    models_4_input = {
        'v2_enhanced': 'model_v2_enhanced.keras',
        'ncf': 'model_ncf.keras',
        'dcn': 'model_dcn.keras'
    }
    predictions_4_input = {}
    
    for name, path in models_4_input.items():
        model_path = os.path.join(OUTPUT_DIR, path)
        if not os.path.exists(model_path):
            print(f"Skipping {name}")
            continue
        
        print(f"Predicting {name}...")
        model = load_model(model_path, custom_objects={'CrossLayer': CrossLayer})
        preds = model.predict(X_test_4_input, batch_size=8192)
        predictions_4_input[name] = preds.flatten()
        del model, preds
        gc.collect()
        
    joblib.dump(predictions_4_input, OUTPUT_DIR + 'preds_4_input.joblib')
    print("Cell 14 Completed")

except Exception as e:
    print(f"Error Cell 14: {e}")
finally:
    try:
        del X_test_4_input, test_df, genre_columns, predictions_4_input
    except NameError:
        pass
    gc.collect()

Predicting 4-Input Models...
Predicting v2_enhanced...
306/306 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step
Predicting ncf...
306/306 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step
Predicting dcn...
306/306 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step
Cell 14 Completed


In [15]:
try:
    print("Predicting Model V3...")
    class GenomeDataGenerator(Sequence):
        def __init__(self, df, genre_cols, genome_dense_matrix, batch_size, **kwargs):
            super().__init__(**kwargs)
            self.df = df
            self.genre_cols = genre_cols
            self.genome_dense_matrix = genome_dense_matrix
            self.batch_size = batch_size
            self.indices = self.df.index.tolist()
        def __len__(self):
            return int(np.ceil(len(self.indices) / self.batch_size))
        def __getitem__(self, index):
            start = index * self.batch_size
            end = min((index + 1) * self.batch_size, len(self.indices))
            batch_indices = self.indices[start:end]
            batch_df = self.df.loc[batch_indices]
            batch_movie_idx = batch_df['movie_idx'].values
            batch_genome_data = self.genome_dense_matrix[batch_movie_idx]
            X_batch = {
                'user_input': batch_df['user_idx'].values,
                'movie_input': batch_df['movie_idx'].values,
                'genre_input': batch_df[self.genre_cols].values,
                'year_input': batch_df['year_scaled'].values,
                'genome_input': batch_genome_data
            }
            return X_batch

    test_df = pd.read_parquet(OUTPUT_DIR + 'test.parquet')
    genre_columns = joblib.load(OUTPUT_DIR + 'genre_columns.joblib')
    
    genome_sparse_matrix = sp.load_npz(OUTPUT_DIR + 'genome_sparse_matrix.npz')
    genome_dense_matrix = genome_sparse_matrix.toarray().astype(np.float32)
    del genome_sparse_matrix
    gc.collect()

    test_gen_genome = GenomeDataGenerator(test_df, genre_columns, genome_dense_matrix, batch_size=4096)
    
    model_path = os.path.join(OUTPUT_DIR, 'model_v3.keras')
    if os.path.exists(model_path):
        model = load_model(model_path)
        preds = model.predict(test_gen_genome) 
        np.save(OUTPUT_DIR + 'preds_v3_genome.npy', preds.flatten()[:len(test_df)])
        del model, preds
        gc.collect()
    
    print("Cell 15 Completed")

except Exception as e:
    print(f"Error Cell 15: {e}")
finally:
    try:
        del test_df, genre_columns, genome_dense_matrix, test_gen_genome
    except NameError:
        pass
    gc.collect()

Predicting Model V3...
611/611 ━━━━━━━━━━━━━━━━━━━━ 9s 14ms/step
Cell 15 Completed


In [16]:
try:
    print("Predicting Model V4...")
    class TagDataGenerator(Sequence):
        def __init__(self, df, genre_cols, user_tag_dense_matrix, movie_tag_dense_matrix, batch_size, **kwargs):
            super().__init__(**kwargs) 
            self.df = df
            self.genre_cols = genre_cols
            self.user_tag_dense_matrix = user_tag_dense_matrix
            self.movie_tag_dense_matrix = movie_tag_dense_matrix
            self.batch_size = batch_size
            self.indices = self.df.index.tolist()
        def __len__(self):
            return int(np.ceil(len(self.indices) / self.batch_size))
        def __getitem__(self, index):
            start = index * self.batch_size
            end = min((index + 1) * self.batch_size, len(self.indices))
            batch_indices = self.indices[start:end]
            batch_df = self.df.loc[batch_indices]
            batch_user_idx = batch_df['user_idx'].values
            batch_movie_idx = batch_df['movie_idx'].values
            batch_user_tags = self.user_tag_dense_matrix[batch_user_idx]
            batch_movie_tags = self.movie_tag_dense_matrix[batch_movie_idx]
            X_batch = {
                'user_input': batch_user_idx,
                'movie_input': batch_movie_idx,
                'genre_input': batch_df[self.genre_cols].values,
                'year_input': batch_df['year_scaled'].values,
                'user_tag_input': batch_user_tags,
                'movie_tag_input': batch_movie_tags
            }
            return X_batch

    test_df = pd.read_parquet(OUTPUT_DIR + 'test.parquet')
    genre_columns = joblib.load(OUTPUT_DIR + 'genre_columns.joblib')
    
    user_tag_dense_matrix = np.load(OUTPUT_DIR + 'user_tag_dense_matrix.npy')
    movie_tag_dense_matrix = np.load(OUTPUT_DIR + 'movie_tag_dense_matrix.npy')

    test_gen_tags = TagDataGenerator(test_df, genre_columns, user_tag_dense_matrix, movie_tag_dense_matrix, batch_size=4096)
    
    model_path = os.path.join(OUTPUT_DIR, 'model_v4.keras')
    if os.path.exists(model_path):
        model = load_model(model_path, custom_objects={'CrossLayer': CrossLayer})
        preds = model.predict(test_gen_tags)
        np.save(OUTPUT_DIR + 'preds_v4_tags.npy', preds.flatten()[:len(test_df)])
        del model, preds
        gc.collect()
    
    print("Cell 16 Completed")

except Exception as e:
    print(f"Error Cell 16: {e}")
finally:
    try:
        del test_df, genre_columns, user_tag_dense_matrix, movie_tag_dense_matrix, test_gen_tags
    except NameError:
        pass
    gc.collect()

Predicting Model V4...
611/611 ━━━━━━━━━━━━━━━━━━━━ 7s 11ms/step
Cell 16 Completed


In [17]:
try:
    print("Ensembling Results...")
    y_test = np.load(OUTPUT_DIR + 'y_test.npy')
    predictions = {}
    
    try:
        preds_4_input = joblib.load(OUTPUT_DIR + 'preds_4_input.joblib')
        predictions.update(preds_4_input)
    except FileNotFoundError:
        pass
        
    try:
        preds_v3 = np.load(OUTPUT_DIR + 'preds_v3_genome.npy')
        predictions['v3_genome'] = preds_v3
    except FileNotFoundError:
        pass

    try:
        preds_v4 = np.load(OUTPUT_DIR + 'preds_v4_tags.npy')
        predictions['v4_tags'] = preds_v4
    except FileNotFoundError:
        pass

    if predictions:
        pred_df = pd.DataFrame(predictions)
        valid_models = list(predictions.keys())
        pred_df['ensemble_avg'] = pred_df[valid_models].mean(axis=1)
        
        print("RESULTS:")
        results_summary = {}
        for col in pred_df.columns:
            min_len = min(len(y_test), len(pred_df[col]))
            rmse = np.sqrt(mean_squared_error(y_test[:min_len], pred_df[col][:min_len]))
            mae = mean_absolute_error(y_test[:min_len], pred_df[col][:min_len])
            results_summary[col] = {'RMSE': rmse, 'MAE': mae}
            print(f"{col.upper():<15} | RMSE: {rmse:.5f} | MAE: {mae:.5f}")
        
        best_model = min(results_summary, key=lambda k: results_summary[k]['RMSE'])
        print(f"BEST MODEL: {best_model.upper()}")
    else:
        print("No predictions found.")

except Exception as e:
    print(f"Error Cell 17: {e}")
finally:
    try:
        del y_test, predictions, pred_df, results_summary
    except NameError:
        pass
    gc.collect()

Ensembling Results...
RESULTS:
V2_ENHANCED     | RMSE: 0.77598 | MAE: 0.58560
NCF             | RMSE: 0.77611 | MAE: 0.58687
DCN             | RMSE: 0.77625 | MAE: 0.58795
V3_GENOME       | RMSE: 0.76260 | MAE: 0.57310
V4_TAGS         | RMSE: 0.76412 | MAE: 0.57447
ENSEMBLE_AVG    | RMSE: 0.75140 | MAE: 0.56734
BEST MODEL: ENSEMBLE_AVG


In [18]:
print("--- RECOMENDATION SYSTEM ---")
try:
    import pandas as pd
    import numpy as np
    import tensorflow as tf
    from tensorflow.keras.models import load_model
    import joblib
    import os

    OUTPUT_DIR = '/kaggle/working/'
    BASE_PATH = '../input/movielens-25m-dataset/ml-25m/'
    
    TARGET_USER_ID = 123 
    
    print(f"Target User ID set to: {TARGET_USER_ID}")

    print("Loading Artifacts...")
    model = load_model(OUTPUT_DIR + 'model_v2_enhanced.keras') 
    user_encoder = joblib.load(OUTPUT_DIR + 'user_encoder.joblib')
    movie_encoder = joblib.load(OUTPUT_DIR + 'movie_encoder.joblib')
    genre_columns = joblib.load(OUTPUT_DIR + 'genre_columns.joblib')
    scaler = joblib.load(OUTPUT_DIR + 'year_scaler.joblib')
    mlb = joblib.load(OUTPUT_DIR + 'mlb_encoder.joblib')

    print("Loading Data...")
    movies_df = pd.read_csv(BASE_PATH + 'movies.csv')
    ratings_df = pd.read_csv(BASE_PATH + 'ratings.csv')
    
    movies_df['year'] = movies_df['title'].str.extract(r'\((\d{4})\)')
    movies_df['year'] = pd.to_numeric(movies_df['year'], errors='coerce').fillna(2000).astype(int)
    movies_df['year_scaled'] = scaler.transform(movies_df[['year']])
    movies_df['genres_list'] = movies_df['genres'].apply(lambda x: x.split('|'))
    genres_encoded = mlb.transform(movies_df['genres_list'])
    genres_df = pd.DataFrame(genres_encoded, columns=genre_columns)
    movies_full = pd.concat([movies_df, genres_df], axis=1)
    
    valid_movie_ids = set(movie_encoder.classes_)
    movies_full = movies_full[movies_full['movieId'].isin(valid_movie_ids)]
    movies_full['movie_idx'] = movie_encoder.transform(movies_full['movieId'])

    def recommend_movies(user_id_input):
        if user_id_input not in user_encoder.classes_:
            print(f"User ID {user_id_input} does not exist.")
            return

        user_idx = user_encoder.transform([user_id_input])[0]
        
        watched_movies = ratings_df[ratings_df['userId'] == user_id_input]['movieId'].values
        
        candidate_movies = movies_full[~movies_full['movieId'].isin(watched_movies)].copy()
        
        num_candidates = len(candidate_movies)
        user_idx_input = np.full(num_candidates, user_idx)
        
        X_pred = {
            'user_input': user_idx_input,
            'movie_input': candidate_movies['movie_idx'].values,
            'genre_input': candidate_movies[genre_columns].values,
            'year_input': candidate_movies['year_scaled'].values
        }
        
        print(f"Predicting for User {user_id_input}...")
        pred_ratings = model.predict(X_pred, batch_size=8192, verbose=0).flatten()
        candidate_movies['predicted_rating'] = pred_ratings
        
        top_10 = candidate_movies.sort_values('predicted_rating', ascending=False).head(10)
        
        print(f"\nTOP 10 RECOMMENDATIONS FOR USER {user_id_input}:")
        print("-" * 60)
        for i, row in enumerate(top_10.itertuples(), 1):
            print(f"{i}. {row.title} ({row.genres}) - Score: {row.predicted_rating:.2f}")
        print("-" * 60)

    recommend_movies(TARGET_USER_ID)

except Exception as e:
    print(f"Error Cell 18: {e}")

--- RECOMENDATION SYSTEM ---
Target User ID set to: 123
Loading Artifacts...
Loading Data...
Predicting for User 123...

TOP 10 RECOMMENDATIONS FOR USER 123:
------------------------------------------------------------
1. The Blue Planet (2001) (Documentary) - Score: 4.59
2. Planet Earth (2006) (Documentary) - Score: 4.58
3. Wallace & Gromit: The Wrong Trousers (1993) (Animation|Children|Comedy|Crime) - Score: 4.58
4. Wallace & Gromit: A Close Shave (1995) (Animation|Children|Comedy) - Score: 4.57
5. Planet Earth II (2016) (Documentary) - Score: 4.56
6. Wallace & Gromit: The Best of Aardman Animation (1996) (Adventure|Animation|Comedy) - Score: 4.55
7. Fargo (1996) (Comedy|Crime|Drama|Thriller) - Score: 4.52
8. Pride and Prejudice (1995) (Drama|Romance) - Score: 4.51
9. Shoah (1985) (Documentary|War) - Score: 4.50
10. Life (2009) (Documentary) - Score: 4.47
------------------------------------------------------------


In [19]:
try:
    print("--- Bắt đầu đóng gói dữ liệu cho Demo ---")
    import pandas as pd
    import numpy as np
    import joblib
    import gc

    OUTPUT_DIR = '/kaggle/working/'
    BASE_PATH = '../input/movielens-25m-dataset/ml-25m/'

    print("1. Đang gom các Encoders...")
    user_encoder = joblib.load(OUTPUT_DIR + 'user_encoder.joblib')
    movie_encoder = joblib.load(OUTPUT_DIR + 'movie_encoder.joblib')
    genre_columns = joblib.load(OUTPUT_DIR + 'genre_columns.joblib')
    scaler = joblib.load(OUTPUT_DIR + 'year_scaler.joblib')
    mlb = joblib.load(OUTPUT_DIR + 'mlb_encoder.joblib')

    print("2. Đang xử lý và đóng gói dữ liệu Phim...")
    movies_df = pd.read_csv(BASE_PATH + 'movies.csv')
    movies_df['year'] = movies_df['title'].str.extract(r'\((\d{4})\)')
    movies_df['year'] = pd.to_numeric(movies_df['year'], errors='coerce').fillna(2000).astype(int)
    movies_df['year_scaled'] = scaler.transform(movies_df[['year']])
    
    movies_df['genres_list'] = movies_df['genres'].apply(lambda x: x.split('|'))
    genres_encoded = mlb.transform(movies_df['genres_list'])
    genres_df = pd.DataFrame(genres_encoded, columns=genre_columns)
    
    movies_full = pd.concat([movies_df, genres_df], axis=1)
    
    valid_movie_ids = set(movie_encoder.classes_)
    movies_full = movies_full[movies_full['movieId'].isin(valid_movie_ids)]
    movies_full['movie_idx'] = movie_encoder.transform(movies_full['movieId'])
    
    movie_genre_features = movies_full[genre_columns].values.astype(np.int8)
    
    demo_movies_df = movies_full[['movieId', 'title', 'genres', 'movie_idx', 'year_scaled']].copy()

    print("3. Đang tạo bản đồ lịch sử xem (User History Map)...")
    ratings_df = pd.read_csv(BASE_PATH + 'ratings.csv', usecols=['userId', 'movieId'])
    user_watched_map = ratings_df.groupby('userId')['movieId'].apply(set).to_dict()
    
    print("4. Đang lưu file 'demo_artifacts.joblib'...")
    artifacts = {
        'user_encoder': user_encoder,
        'movies_df': demo_movies_df,
        'movie_genre_features': movie_genre_features,
        'user_watched_map': user_watched_map
    }
    
    joblib.dump(artifacts, OUTPUT_DIR + 'demo_artifacts.joblib', compress=3)
    print(f"--- XONG! Hãy tải file 'demo_artifacts.joblib' về máy ---")

except Exception as e:
    print(f"LỖI Cell 19: {e}")
finally:
    try:
        del ratings_df, movies_full, movies_df, genres_df
    except NameError:
        pass
    gc.collect()

--- Bắt đầu đóng gói dữ liệu cho Demo ---
1. Đang gom các Encoders...
2. Đang xử lý và đóng gói dữ liệu Phim...
3. Đang tạo bản đồ lịch sử xem (User History Map)...
4. Đang lưu file 'demo_artifacts.joblib'...
--- XONG! Hãy tải file 'demo_artifacts.joblib' về máy ---
